# Explore KITTI MOT

Sanity-check notebook (task T2). Confirms the loader handles every sequence, the class distribution looks like published KITTI, and ground-truth boxes overlay the right pixels.

## Data card

**Dataset:** KITTI 2D MOT, training split (21 sequences with public GT). Sequence-level holdout — 16 sequences for train, 5 for val. No frame leakage.

**Train sequences:** 0000, 0002, 0003, 0004, 0005, 0007, 0008, 0009, 0010, 0011, 0012, 0014, 0015, 0016, 0018, 0020

**Val sequences:** 0001, 0006, 0013, 0017, 0019

**Classes:** Car, Pedestrian, Cyclist. `DontCare` and other KITTI categories are filtered out at load time (see [`tracking.data.kitti.parse_label_file`](../src/tracking/data/kitti.py)).

**Source:** [cvlibs.net/datasets/kitti/eval_tracking.php](https://www.cvlibs.net/datasets/kitti/eval_tracking.php)

**License:** KITTI non-commercial — research use only. Public dataset; nothing private flows through this repo.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from tracking.data.kitti import KITTI_CLASSES, KittiTrackingDataset

sns.set_theme(style="whitegrid", context="notebook")

DATA_ROOT = Path("../data/kitti_tracking")
TRAIN_SEQS = [f"{i:04d}" for i in [0, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 14, 15, 16, 18, 20]]
VAL_SEQS = [f"{i:04d}" for i in [1, 6, 13, 17, 19]]

train = KittiTrackingDataset.from_split(DATA_ROOT, TRAIN_SEQS)
val = KittiTrackingDataset.from_split(DATA_ROOT, VAL_SEQS)

print(f"Train: {len(train.sequences)} seqs, {train.total_frames():>5} frames, {train.total_annotations():>6} annotations")
print(f"Val:   {len(val.sequences)} seqs, {val.total_frames():>5} frames, {val.total_annotations():>6} annotations")

## Per-sequence summary

If any sequence has a wildly different frame count or class mix, this is where you'd spot it. KITTI 0019 is historically pedestrian-heavy — useful to flag because tracker hyperparameters tuned on car-dominated sequences may underperform there.

In [ ]:
def summarise(ds: KittiTrackingDataset, split: str) -> pd.DataFrame:
    rows = []
    for s in ds.sequences:
        counts = Counter(a.obj_class for a in s.annotations)
        rows.append({
            "split": split,
            "seq": s.name,
            "frames": s.num_frames,
            "annotations": len(s.annotations),
            **{c: counts.get(c, 0) for c in KITTI_CLASSES},
        })
    return pd.DataFrame(rows)

summary = pd.concat([summarise(train, "train"), summarise(val, "val")], ignore_index=True)
summary

## Class distribution

Headline imbalance check. KITTI is car-dominated; pedestrians and cyclists are the long tail. The eval headline (HOTA on cars) won't necessarily reflect performance on the rarer classes — worth keeping in mind when reading the final results table.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
by_class = summary.groupby("split")[list(KITTI_CLASSES)].sum().T
by_class.plot.bar(ax=ax, color=["#4c72b0", "#dd8452"], edgecolor="white")
ax.set_ylabel("annotation count")
ax.set_title("Class distribution by split")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", fontsize=8, padding=2)
plt.tight_layout()
plt.show()

totals = by_class.sum(axis=1)
print("Class share (train+val):")
for cls, count in totals.items():
    print(f"  {cls:<11} {count:>6}  ({count / totals.sum():.1%})")

## Track length distribution

How many frames each ground-truth track is alive for. The very short tracks are the ones a tracker is most likely to fragment — they correspond to occlusions, frame-edge exits, and (sometimes) GT noise.

**Why this matters for tuning later:** ByteTrack's `track_buffer` controls how long a lost track is kept alive waiting for re-association. If the median track length is much longer than the buffer, you'll over-fragment; if much shorter, you'll let dead tracks linger and IDsw will spike. The median + p95 below give a starting range to think about.

In [ ]:
track_lengths: dict[str, list[int]] = {c: [] for c in KITTI_CLASSES}
for s in train.sequences:
    by_track: dict[tuple[str, int], int] = {}
    for a in s.annotations:
        key = (a.obj_class, a.track_id)
        by_track[key] = by_track.get(key, 0) + 1
    for (cls, _tid), length in by_track.items():
        track_lengths[cls].append(length)

fig, axes = plt.subplots(1, len(KITTI_CLASSES), figsize=(13, 3.5), sharey=False)
for ax, cls in zip(axes, KITTI_CLASSES, strict=True):
    lengths = track_lengths[cls]
    if not lengths:
        ax.set_title(f"{cls} (none)")
        continue
    sns.histplot(lengths, bins=40, ax=ax, color="#4c72b0", edgecolor="white")
    ax.set_title(f"{cls} (n={len(lengths)})")
    ax.set_xlabel("frames per track")
    ax.set_yscale("log")
axes[0].set_ylabel("track count (log)")
plt.tight_layout()
plt.show()

print(f"{'class':<11} {'n':>5} {'median':>7} {'p95':>5} {'max':>5}")
for cls in KITTI_CLASSES:
    lengths = track_lengths[cls]
    if not lengths:
        continue
    arr = np.array(lengths)
    print(f"{cls:<11} {len(arr):>5} {int(np.median(arr)):>7} {int(np.percentile(arr, 95)):>5} {arr.max():>5}")

## GT-overlaid sample frames

Three frames hand-picked to span the difficulty spectrum. If boxes don't line up tightly with objects, parsing or coordinate handling is wrong.

- **Crowded urban (val seq 0006, frame 150):** dense pedestrian scene — the kind where ByteTrack's two-stage matching is supposed to earn its keep.
- **Sparse highway (val seq 0001, frame 10):** few large fast-moving cars — clean detection, association is mostly trivial.
- **Cyclist-heavy (train seq 0011, frame 100):** mixed-class scene including the rare Cyclist class — visual check that we're not silently dropping it.

In [ ]:
CLASS_COLORS_BGR = {
    "Car": (0, 200, 0),
    "Pedestrian": (255, 80, 0),
    "Cyclist": (0, 80, 255),
}


def render_frame(seq, frame: int) -> np.ndarray:
    img_path = seq.image_dir / f"{frame:06d}.png"
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(img_path)
    for a in seq.annotations_for_frame(frame):
        color = CLASS_COLORS_BGR[a.obj_class]
        p1, p2 = (int(a.x1), int(a.y1)), (int(a.x2), int(a.y2))
        cv2.rectangle(img, p1, p2, color, 2)
        label = f"{a.obj_class[:3]} #{a.track_id}"
        cv2.putText(img, label, (p1[0], p1[1] - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


samples = [
    ("0006", 150, "Crowded urban"),
    ("0001", 10, "Sparse highway"),
    ("0011", 100, "Cyclist-heavy"),
]
by_name = {s.name: s for s in (*train.sequences, *val.sequences)}

fig, axes = plt.subplots(len(samples), 1, figsize=(13, 3.5 * len(samples)))
for ax, (seq_name, frame, label) in zip(axes, samples, strict=True):
    seq = by_name[seq_name]
    n_boxes = len(seq.annotations_for_frame(frame))
    ax.imshow(render_frame(seq, frame))
    ax.set_title(f"{label} \u2014 seq {seq_name}, frame {frame} ({n_boxes} GT boxes)")
    ax.axis("off")
plt.tight_layout()
plt.show()